In [1]:
"""
================================================================================
CELL 0 — PACKAGE IMPORTS / INSTALLS
Run this first, before Cell 1 (data) and Cell 2 (Q6 model).
torch/pandas/numpy/scikit-learn are already present on Colab/Kaggle by default
-- only pennylane (quantum circuit correctness verification) and kagglehub
(dataset download) typically need installing.
================================================================================
"""

import subprocess, sys

def pip_install(*pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    except subprocess.CalledProcessError:
        # some environments (externally-managed Python) refuse a bare install -- retry with the override.
        # Harmless on Colab/Kaggle, where this branch is never needed.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs], check=True)

pip_install("pennylane", "kagglehub")

# sanity import check -- fails loudly here rather than mid-run later
import math, os, glob, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pennylane as qml
import kagglehub

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("pennylane:", qml.__version__)
print("pandas:", pd.__version__, "| numpy:", np.__version__)
print("All packages imported OK.")

torch: 2.11.0+cpu | cuda available: False
pennylane: 0.45.1
pandas: 2.2.3 | numpy: 2.1.3
All packages imported OK.


In [2]:
"""
================================================================================
CELL 1 — DATA ACQUISITION + PROCESSING
Same methodology as all prior cells (Kaggle download, phi-unit detection,
target construction, standardization, graph construction) with ONE change:
SPLIT is now 20% train / 20% val / 60% test -- the 60% that was previously
discarded as "unused" is now the test set instead. Nothing is thrown away.
================================================================================
"""

import os, math, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# ==============================================================================
# 0. CONFIGURATION  (unchanged from all prior cells)
# ==============================================================================
SEED = 42 # 1 60 100 120
USE_PT_FLOOR = True
PHI_UNITS = "auto"

KAGGLE_DATASET = "prateekagnihotri/cms-trigger-dataset"
LOCAL_FALLBACK_DIR = "/content"

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

N_NODES = 4
N_FEATS = 7
D_NODE = 4
N_QUBITS = 4
N_EDGES = N_NODES * (N_NODES - 1)
QDIM = 2 ** N_QUBITS
assert D_NODE == N_QUBITS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ==============================================================================
# 1. DATA ACQUISITION  (unchanged)
# ==============================================================================
STATIONS = [0, 2, 3, 4]
STATION_FEATS = ["Phi", "Theta", "BendingAngle", "TimeInfo", "RingNumber", "Front", "Mask"]
REQUIRED_STATION_COLS = [f"{feat}_{st}" for st in STATIONS for feat in STATION_FEATS]
REQUIRED_COLS = REQUIRED_STATION_COLS + ["q/pt"]

def find_csv_with_required_columns(root_dir, required_cols, max_files=200):
    for path in glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)[:max_files]:
        try:
            header = pd.read_csv(path, nrows=0).columns.tolist()
            if all(c in header for c in required_cols):
                return path
        except Exception:
            continue
    return None

def download_dataset():
    try:
        import kagglehub
        path = kagglehub.dataset_download(KAGGLE_DATASET)
        print("kagglehub download path:", path)
        found = find_csv_with_required_columns(path, REQUIRED_COLS)
        if found:
            return found
        any_csv = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
        if any_csv:
            return any_csv[0]
    except Exception as e:
        print("kagglehub path unavailable or failed:", repr(e))
    try:
        import kaggle
        dl_dir = "/content/cms_trigger_dataset"
        os.makedirs(dl_dir, exist_ok=True)
        kaggle.api.dataset_download_files(KAGGLE_DATASET, path=dl_dir, unzip=True)
        found = find_csv_with_required_columns(dl_dir, REQUIRED_COLS)
        if found:
            return found
    except Exception as e:
        print("kaggle API path unavailable or failed:", repr(e))
    found = find_csv_with_required_columns(LOCAL_FALLBACK_DIR, REQUIRED_COLS)
    if found:
        return found
    raise FileNotFoundError(f"Could not locate {KAGGLE_DATASET} CSV automatically.")

csv_path = download_dataset()
print("Discovered CSV file:", csv_path)
df_raw = pd.read_csv(csv_path)
missing = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")
print("Original dataset size:", len(df_raw))

# ==============================================================================
# 2. PHI UNIT DETECTION + TARGET CONSTRUCTION  (unchanged)
# ==============================================================================
_phi_cols = [f"Phi_{st}" for st in STATIONS]
_phi_vals = pd.to_numeric(df_raw[_phi_cols].stack(), errors="coerce")
_phi_vals = _phi_vals[np.isfinite(_phi_vals)]
_phi_span = float(_phi_vals.max() - _phi_vals.min())
if PHI_UNITS == "auto":
    PHI_IS_RADIANS = _phi_span <= 2 * math.pi + 1e-6
elif PHI_UNITS == "radians":
    PHI_IS_RADIANS = True
else:
    PHI_IS_RADIANS = False
print("Phi treated as:", "radians (wrapped)" if PHI_IS_RADIANS else "integer bins (plain diff)")

df = df_raw.copy()
with np.errstate(divide="ignore", invalid="ignore"):
    df["pT"] = np.abs(1.0 / df["q/pt"])

invalid_mask = (
    df["pT"].isna() | np.isinf(df["pT"]) | (df["pT"] <= 0)
    | df[REQUIRED_STATION_COLS].isna().any(axis=1)
    | np.isinf(df[REQUIRED_STATION_COLS]).any(axis=1)
)
df = df.loc[~invalid_mask].reset_index(drop=True)
print("Remaining rows after cleaning:", len(df))

# ==============================================================================
# 3. SPLIT — CHANGED: 20% train / 20% val / 60% test, nothing discarded
# ==============================================================================
N_total = len(df)
train_df, _rest1 = train_test_split(df, train_size=0.20, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(_rest1, train_size=0.25, random_state=SEED, shuffle=True)
# _rest1 is 80% of total; 0.25 of that = 20% of total -> val. The remaining 75% of
# _rest1 (=60% of total) is test_df in full -- no further split, no unused portion.
print(f"Total rows: {N_total} | Train: {len(train_df)} ({100*len(train_df)/N_total:.1f}%) "
      f"| Val: {len(val_df)} ({100*len(val_df)/N_total:.1f}%) "
      f"| Test: {len(test_df)} ({100*len(test_df)/N_total:.1f}%)")
assert len(train_df) + len(val_df) + len(test_df) == N_total, "split must cover the full dataset now"

# ==============================================================================
# 4. PREPROCESSING  (unchanged)
# ==============================================================================
def station_cols(st):
    return [f"{feat}_{st}" for feat in STATION_FEATS]

def build_X(d):
    return np.stack([d[station_cols(st)].values for st in STATIONS], axis=1).astype(np.float32)

X_train_raw = build_X(train_df)
X_val_raw   = build_X(val_df)
X_test_raw  = build_X(test_df)

X_mu = X_train_raw.mean(axis=0, keepdims=True)
X_sigma = np.maximum(X_train_raw.std(axis=0, keepdims=True), 1e-6)

def standardize(x):
    return (x - X_mu) / X_sigma

X_train, X_val, X_test = standardize(X_train_raw), standardize(X_val_raw), standardize(X_test_raw)

PHI_IDX = STATION_FEATS.index("Phi")
BA_IDX = STATION_FEATS.index("BendingAngle")
phi_train, ba_train = X_train_raw[:, :, PHI_IDX], X_train_raw[:, :, BA_IDX]
phi_val, ba_val     = X_val_raw[:, :, PHI_IDX], X_val_raw[:, :, BA_IDX]
phi_test, ba_test   = X_test_raw[:, :, PHI_IDX], X_test_raw[:, :, BA_IDX]

y_train = np.log1p(train_df["pT"].values.astype(np.float32))
y_val   = np.log1p(val_df["pT"].values.astype(np.float32))
y_test  = np.log1p(test_df["pT"].values.astype(np.float32))
pT_train = train_df["pT"].values.astype(np.float32)
pT_test  = test_df["pT"].values.astype(np.float32)

Y_MEAN = float(y_train.mean())
Y_STD  = float(y_train.std() + 1e-8)
PT_FLOOR_TRAIN = float(train_df["pT"].min())
PT_CLAMP_LO = PT_FLOOR_TRAIN if USE_PT_FLOOR else 0.0
print(f"Target log(1+pT): train mean={Y_MEAN:.4f} std={Y_STD:.4f}")

# ==============================================================================
# 5. GRAPH CONSTRUCTION  (unchanged)
# ==============================================================================
EDGES = [(i, j) for i in range(N_NODES) for j in range(N_NODES) if j != i]
edge_recv = torch.tensor([e[0] for e in EDGES], dtype=torch.long)
edge_send = torch.tensor([e[1] for e in EDGES], dtype=torch.long)
assert len(EDGES) == 12

def compute_r(phi, ba, recv, send):
    phi_i, phi_j = phi[:, recv], phi[:, send]
    ba_i, ba_j = ba[:, recv], ba[:, send]
    if PHI_IS_RADIANS:
        dphi = np.arctan2(np.sin(phi_j - phi_i), np.cos(phi_j - phi_i))
    else:
        dphi = phi_j - phi_i
    return np.stack([np.abs(ba_j - ba_i), np.abs(dphi)], axis=-1)

r_train_np = compute_r(phi_train, ba_train, edge_recv.numpy(), edge_send.numpy())
r_mean = torch.tensor(r_train_np.reshape(-1, 2).mean(axis=0), dtype=torch.float32)
r_std  = torch.tensor(np.maximum(r_train_np.reshape(-1, 2).std(axis=0), 1e-6), dtype=torch.float32)
r_mean_d, r_std_d = r_mean.to(DEVICE), r_std.to(DEVICE)

print("\nData ready: X_train/X_val/X_test, phi_*/ba_* , y_*, pT_train/pT_test, "
      "edge_recv/edge_send, r_mean/r_std (+ _d device copies), Y_MEAN/Y_STD, PT_CLAMP_LO.")

Device: cpu


100%|██████████| 50.6M/50.6M [00:02<00:00, 18.1MB/s]

Extracting files...


kagglehub download path: /root/.cache/kagglehub/datasets/prateekagnihotri/cms-trigger-dataset/versions/1
Discovered CSV file: /root/.cache/kagglehub/datasets/prateekagnihotri/cms-trigger-dataset/versions/1/CMS_trigger.csv
Original dataset size: 1179356
Phi treated as: integer bins (plain diff)
Remaining rows after cleaning: 1179356
Total rows: 1179356 | Train: 235871 (20.0%) | Val: 235871 (20.0%) | Test: 707614 (60.0%)
Target log(1+pT): train mean=2.0636 std=0.9417

Data ready: X_train/X_val/X_test, phi_*/ba_* , y_*, pT_train/pT_test, edge_recv/edge_send, r_mean/r_std (+ _d device copies), Y_MEAN/Y_STD, PT_CLAMP_LO.


In [6]:
"""
================================================================================
CELL 3 -- PI-QGAT, 144-parameter model (quantum interaction block)          [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0 (installs) and Cell 1 (data). Independent of Cell 2.

WHAT THIS CELL DOES
  * Trains the Physics-Informed Quantum Graph Attention Network (PI-QGAT) described below, 5 seeds x 50 epochs, prints
    every epoch, restores the best-validation checkpoint, then evaluates on the 60% test split.
  * Reports log1p(pT) metrics (R2, MAE, MSE, RMSE), pT metrics in GeV (MAE, MSE, RMSE; no R2), a per-pT-class table
    (0-2, 2-4, 4-6, 6-10, 10-20, 20-50, 50-100, >=100 GeV), convergence statistics, training time and INFERENCE time.
  * Verifies the fast circuit against PennyLane before training.

MODEL (144 trainable parameters)
   node_proj 7->4 (32) | pair_proj 8->4 (36) | physics_prior 2->1 (3) | head 4->6->1 (37)
   quantum block: 16 circuit angles + entangle_gate1 4->4 (20, makes the RZZ angles data dependent) = 36
   The old model had 132 parameters; its 2nd RZZ layer (entangle_gate2, 20 params) is a diagonal gate before a
   Z-basis measurement and provably cannot change <Z>, so it was dead weight and is removed (same function).
   The head 4->6->1 (+32 params vs the original 4->1 head) is the only architectural change vs the original cell.

SPEED / SIZE MEASURES (none changes the math)
   whole epoch in one XLA call (lax.scan), exact real-arithmetic 4-qubit simulator, one fused 32x32 matrix per
   variational layer, top-half product for the real input state, inference-time constant folding of the two layer
   matrices ("folded" path, checked equal to the training graph), float16 weight-storage check (144 params = 288 bytes).

NOTE ON RUN TIME: 5 seeds x 50 epochs. Reduce SEEDS for a quick look (e.g. SEEDS = [1]).
================================================================================
"""
SEEDS = [SEED]
VARIANTS_TO_RUN = ["quantum"]
RESULTS_FILE = "piqgat_results_quantum.json"

# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, makes the RZZ angles)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s (train {t_tr:.1f} + eval {sec - t_tr:.1f}){flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

# ==============================================================================
# 9. RUN
# ==============================================================================
ALL = {}
for variant in VARIANTS_TO_RUN:
    P_demo = init_params(variant, 0)
    rows, total = count_table(P_demo)
    print("\n" + "#" * 96 + f"\n# VARIANT: {variant}   |   {total} trainable parameters\n" + "#" * 96)
    for nm, n in rows:
        print(f"   {nm:<38}{n:>6}")
    print(f"   {'TOTAL':<38}{total:>6}")
    if variant == "quantum":
        verify_against_pennylane(); print_circuit()
    C = Compiled(variant)
    res = []
    for seed in SEEDS:
        print(f"\n--- {variant} | seed {seed} ---")
        tr = train_one(C, seed)
        m, bins, tm = evaluate_test(C, tr, seed)
        h = tr["hist"]
        print(f"   >> best epoch {tr['best_ep']} (val loss {h[tr['best_ep']-1]['val_loss']:.4f}) | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f} | TEST pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.2f}  RMSE {m['rmse_pt']:.3f}")
        print(f"   >> inference (full {len(X_test)}-row test pass, data on device): "
              + "  ".join(f"{k}: {v['test_pass_s']*1e3:.0f} ms = {v['us_per_row']:.2f} us/row ({v['rows_per_s']/1e3:.0f}k rows/s)"
                          for k, v in tm.items() if not k.startswith("latency")))
        print(f"   >> latency ms @ batch 1/64/1024: " + "   ".join(k[8:] + " " + " / ".join(f"{x:.2f}" for x in v.values()) for k, v in tm.items() if k.startswith("latency"))
              + f"   | fp16-weights dR2_log {m['fp16_dR2_log']:+.5f}" + (f" | folded-vs-graph max|d| {m['fold_maxdiff']:.1e}" if 'fold_maxdiff' in m else ""))
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{variant}_seed{seed}.npz"), w=np.asarray(tr["w"]),
                 w_fp16=np.asarray(tr["w"].astype(jnp.float16)))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        test=m, bins=bins, timing={k: ({str(b): x for b, x in v.items()} if k.startswith("latency") else v) for k, v in tm.items()},
                        hist=h))
    ALL[variant] = res

# ==============================================================================
# 10. AGGREGATE REPORT  (mean +- std over seeds)
# ==============================================================================
def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

def report(variant, res):
    print("\n" + "=" * 96 + f"\nFINAL REPORT  |  {variant}  |  {res[0]['n_par']} params  |  {len(res)} seeds {[r['seed'] for r in res]}  |  best-val checkpoint\n" + "=" * 96)
    g = lambda key: [r["test"][key] for r in res]
    print("TEST SET, log1p(pT) scale:")
    for lab, key in [("R2", "r2_log"), ("MAE", "mae_log"), ("MSE", "mse_log"), ("RMSE", "rmse_log")]:
        print(f"   {lab:<6}{ms(g(key), '{:.5f}')}")
    print("TEST SET, pT in GeV (R2 intentionally omitted):")
    for lab, key, f in [("MAE (GeV)", "mae_pt", "{:.4f}"), ("MSE (GeV^2)", "mse_pt", "{:.3f}"), ("RMSE (GeV)", "rmse_pt", "{:.4f}")]:
        print(f"   {lab:<12}{ms(g(key), f)}")
    print(f"   predictions clamped at pT floor: {ms(g('n_clamped'), '{:.0f}')} rows")
    print("\nCONVERGENCE (val R2_log mean +- std at epoch) and TRAINING TIME:")
    for ep in sorted({1, 2, 5, 10, 25, EPOCHS}):
        if ep <= min(len(r["hist"]) for r in res):
            print(f"   epoch {ep:>2}: {ms([r['hist'][ep-1]['val_r2_log'] for r in res])}   (val loss {ms([r['hist'][ep-1]['val_loss'] for r in res], '{:.4f}')})")
    e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), None) for r in res]
    print(f"   epochs to reach within 2% of own best val loss: {ms(e2, '{:.1f}')}   | best epoch: {ms([r['best_ep'] for r in res], '{:.1f}')}")
    print(f"   sec/epoch (median over epochs, incl. val pass): {ms([np.median([h['sec'] for h in r['hist']]) for r in res], '{:.2f}')}"
          f"   | total train: {ms([r['train_total_s'] for r in res], '{:.0f}')} s")
    print("\nINFERENCE TIMING (test pass over all test rows, data already on device, median of 3):")
    for k in res[0]["timing"]:
        if k.startswith("latency"): continue
        print(f"   {k:<7} path: {ms([r['timing'][k]['test_pass_s'] * 1e3 for r in res], '{:.0f}')} ms/pass   "
              f"{ms([r['timing'][k]['us_per_row'] for r in res], '{:.2f}')} us/row   {ms([r['timing'][k]['rows_per_s'] / 1e3 for r in res], '{:.0f}')}k rows/s")
    for k in res[0]["timing"]:
        if k.startswith("latency"):
            print(f"   latency ms, {k[8:]} path (batch 1 / 64 / 1024): " + " / ".join(ms([r['timing'][k][b] for r in res], '{:.2f}') for b in ("1", "64", "1024")))
    print(f"   model size: {res[0]['n_par']} params = {4 * res[0]['n_par']} bytes float32, {2 * res[0]['n_par']} bytes float16 "
          f"(fp16-rounded weights change test R2_log by {ms(g('fp16_dR2_log'), '{:+.5f}')})")
    print("\nPER pT CLASS (test, mean over seeds; +- is std of MAE_pT):")
    print(f"{'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>16}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
    for i, b0 in enumerate(res[0]["bins"]):
        col = lambda k: [r["bins"][i][k] for r in res]
        s = np.std(col("mae_pt"), ddof=1) if len(res) > 1 else 0.0
        print(f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{b0['frac']:>6.1f}%{np.mean(col('mae_log')):>9.4f}"
              f"{np.mean(col('mae_pt')):>10.3f}+-{s:<5.3f}{np.mean(col('mse_pt')):>13.2f}{np.mean(col('rmse_pt')):>10.3f}"
              f"{np.mean(col('med_rel')):>10.3f}{np.mean(col('bias')):>8.3f}{np.mean(col('sse_share')):>6.1f}%")

for v in VARIANTS_TO_RUN:
    report(v, ALL[v])

out_path = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump({v: ALL[v] for v in ALL}, open(out_path, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {out_path}")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['quantum']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# VARIANT: quantum   |   144 trainable parameters
################################################################################################
   node_proj 7->4                            32
   physics_prior 2->1                         3
   head 4->6                                 30
   head 6->1                                  7
   pair_proj 8->4                            36
   entangle_gate1 4->4 (RZZ angles)          20
   circuit angles (2x4x2)                    16
   TOTAL                                    144
------------------------------------------------------------------------------------------------
VERIFY: closed-form JAX circuit vs Pen

In [4]:
"""
================================================================================
CELL 4 -- QUANTUM-vs-CLASSICAL ABLATION (same architecture, only the interaction block changes)      [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0 and Cell 1. Independent of Cell 2 and Cell 3 (if Cell 3 saved its results file, they are loaded for the
comparison table at the end; otherwise set RERUN_QUANTUM_REFERENCE = True to retrain the quantum model here).

VARIANTS (everything else -- node_proj, pair_proj, physics prior, attention, message passing, 4->6->1 head, loss,
optimizer, seeds, batch order, init of the shared layers -- is identical to Cell 3)
  classical36 : PARAMETER-MATCHED control. The whole quantum block (16 circuit angles + 20 entangle_gate1 params = 36) is
                replaced by a 36-parameter classical block  z_exp = tanh( W2 tanh(W1 theta + b1) ),  W1, W2 in R^{4x4}, b1 in R^4
                (same 4-in / 4-out interface, same [-1,1] output range). Total 144 params = identical to the quantum model.
  classical16 : the 16 circuit angles are replaced by a 16-parameter classical map  z_exp = tanh(W theta), W in R^{4x4}.
                entangle_gate1 (the 20 RZZ-angle parameters) is removed as requested, so this model has 124 params
                (quantum model: 144 = 124 + 20 RZZ-gate params).
  noquantum   : the whole quantum block is removed (z_exp = 0, so the message is h_j and the attention logit is the physics
                prior only). pair_proj feeds only the block, so it is removed too: 72 params.

HOW TO READ IT: quantum (144) vs classical36 (144) is the equal-parameter test of the circuit itself. classical16 (124) and
noquantum (72) show how much the interaction block and the RZZ generator contribute at all. IMPORTANT: use the same SEEDS here
as in Cell 3 -- the paired comparison only uses seeds present in both.
================================================================================
"""
SEEDS = [42]                              # must match Cell 3 (paired comparison); e.g. [1, 2, 3, 4, 5]
RERUN_QUANTUM_REFERENCE = False          # True -> also retrain the 144-param quantum model inside this cell
VARIANTS_TO_RUN = ["classical36", "classical16", "noquantum"] + (["quantum"] if RERUN_QUANTUM_REFERENCE else [])
RESULTS_FILE = "piqgat_results_ablation.json"

# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, sets the RZZ angles)
#              classical36 : tanh(W2 tanh(W1 theta + b1)), 16+4+16 = 36 params (matches the quantum block's 36)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical36":
        l1 = _init_lin(ks[7], N_QUBITS, N_QUBITS); l2 = _init_lin(jax.random.fold_in(ks[7], 1), N_QUBITS, N_QUBITS, bias=False)
        P["c36"] = {"W1": l1["W"], "b1": l1["b"], "W2": l2["W"]}
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        elif variant == "classical36":
            c = P["c36"]
            z = jnp.tanh(jnp.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4", "c36": "classical block 4-4-4 (W1,b1,W2)"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s (train {t_tr:.1f} + eval {sec - t_tr:.1f}){flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

# ==============================================================================
# 9. RUN
# ==============================================================================
ALL = {}
for variant in VARIANTS_TO_RUN:
    P_demo = init_params(variant, 0)
    rows, total = count_table(P_demo)
    print("\n" + "#" * 96 + f"\n# VARIANT: {variant}   |   {total} trainable parameters\n" + "#" * 96)
    for nm, n in rows:
        print(f"   {nm:<38}{n:>6}")
    print(f"   {'TOTAL':<38}{total:>6}")
    if variant == "quantum":
        verify_against_pennylane(); print_circuit()
    C = Compiled(variant)
    res = []
    for seed in SEEDS:
        print(f"\n--- {variant} | seed {seed} ---")
        tr = train_one(C, seed)
        m, bins, tm = evaluate_test(C, tr, seed)
        h = tr["hist"]
        print(f"   >> best epoch {tr['best_ep']} (val loss {h[tr['best_ep']-1]['val_loss']:.4f}) | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f} | TEST pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.2f}  RMSE {m['rmse_pt']:.3f}")
        print(f"   >> inference (full {len(X_test)}-row test pass, data on device): "
              + "  ".join(f"{k}: {v['test_pass_s']*1e3:.0f} ms = {v['us_per_row']:.2f} us/row ({v['rows_per_s']/1e3:.0f}k rows/s)"
                          for k, v in tm.items() if not k.startswith("latency")))
        print(f"   >> latency ms @ batch 1/64/1024: " + "   ".join(k[8:] + " " + " / ".join(f"{x:.2f}" for x in v.values()) for k, v in tm.items() if k.startswith("latency"))
              + f"   | fp16-weights dR2_log {m['fp16_dR2_log']:+.5f}" + (f" | folded-vs-graph max|d| {m['fold_maxdiff']:.1e}" if 'fold_maxdiff' in m else ""))
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{variant}_seed{seed}.npz"), w=np.asarray(tr["w"]),
                 w_fp16=np.asarray(tr["w"].astype(jnp.float16)))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        test=m, bins=bins, timing={k: ({str(b): x for b, x in v.items()} if k.startswith("latency") else v) for k, v in tm.items()},
                        hist=h))
    ALL[variant] = res

# ==============================================================================
# 10. AGGREGATE REPORT  (mean +- std over seeds)
# ==============================================================================
def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

def report(variant, res):
    print("\n" + "=" * 96 + f"\nFINAL REPORT  |  {variant}  |  {res[0]['n_par']} params  |  {len(res)} seeds {[r['seed'] for r in res]}  |  best-val checkpoint\n" + "=" * 96)
    g = lambda key: [r["test"][key] for r in res]
    print("TEST SET, log1p(pT) scale:")
    for lab, key in [("R2", "r2_log"), ("MAE", "mae_log"), ("MSE", "mse_log"), ("RMSE", "rmse_log")]:
        print(f"   {lab:<6}{ms(g(key), '{:.5f}')}")
    print("TEST SET, pT in GeV (R2 intentionally omitted):")
    for lab, key, f in [("MAE (GeV)", "mae_pt", "{:.4f}"), ("MSE (GeV^2)", "mse_pt", "{:.3f}"), ("RMSE (GeV)", "rmse_pt", "{:.4f}")]:
        print(f"   {lab:<12}{ms(g(key), f)}")
    print(f"   predictions clamped at pT floor: {ms(g('n_clamped'), '{:.0f}')} rows")
    print("\nCONVERGENCE (val R2_log mean +- std at epoch) and TRAINING TIME:")
    for ep in sorted({1, 2, 5, 10, 25, EPOCHS}):
        if ep <= min(len(r["hist"]) for r in res):
            print(f"   epoch {ep:>2}: {ms([r['hist'][ep-1]['val_r2_log'] for r in res])}   (val loss {ms([r['hist'][ep-1]['val_loss'] for r in res], '{:.4f}')})")
    e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), None) for r in res]
    print(f"   epochs to reach within 2% of own best val loss: {ms(e2, '{:.1f}')}   | best epoch: {ms([r['best_ep'] for r in res], '{:.1f}')}")
    print(f"   sec/epoch (median over epochs, incl. val pass): {ms([np.median([h['sec'] for h in r['hist']]) for r in res], '{:.2f}')}"
          f"   | total train: {ms([r['train_total_s'] for r in res], '{:.0f}')} s")
    print("\nINFERENCE TIMING (test pass over all test rows, data already on device, median of 3):")
    for k in res[0]["timing"]:
        if k.startswith("latency"): continue
        print(f"   {k:<7} path: {ms([r['timing'][k]['test_pass_s'] * 1e3 for r in res], '{:.0f}')} ms/pass   "
              f"{ms([r['timing'][k]['us_per_row'] for r in res], '{:.2f}')} us/row   {ms([r['timing'][k]['rows_per_s'] / 1e3 for r in res], '{:.0f}')}k rows/s")
    for k in res[0]["timing"]:
        if k.startswith("latency"):
            print(f"   latency ms, {k[8:]} path (batch 1 / 64 / 1024): " + " / ".join(ms([r['timing'][k][b] for r in res], '{:.2f}') for b in ("1", "64", "1024")))
    print(f"   model size: {res[0]['n_par']} params = {4 * res[0]['n_par']} bytes float32, {2 * res[0]['n_par']} bytes float16 "
          f"(fp16-rounded weights change test R2_log by {ms(g('fp16_dR2_log'), '{:+.5f}')})")
    print("\nPER pT CLASS (test, mean over seeds; +- is std of MAE_pT):")
    print(f"{'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>16}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
    for i, b0 in enumerate(res[0]["bins"]):
        col = lambda k: [r["bins"][i][k] for r in res]
        s = np.std(col("mae_pt"), ddof=1) if len(res) > 1 else 0.0
        print(f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{b0['frac']:>6.1f}%{np.mean(col('mae_log')):>9.4f}"
              f"{np.mean(col('mae_pt')):>10.3f}+-{s:<5.3f}{np.mean(col('mse_pt')):>13.2f}{np.mean(col('rmse_pt')):>10.3f}"
              f"{np.mean(col('med_rel')):>10.3f}{np.mean(col('bias')):>8.3f}{np.mean(col('sse_share')):>6.1f}%")

for v in VARIANTS_TO_RUN:
    report(v, ALL[v])

out_path = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump({v: ALL[v] for v in ALL}, open(out_path, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {out_path}")

# ==============================================================================
# 11. QUANTUM vs CLASSICAL COMPARISON  (paired by seed: same init for shared layers, same batch order)
# ==============================================================================
REF = ALL.get("quantum")
if REF is None:
    _p = os.path.join(RESULTS_DIR, "piqgat_results_quantum.json")
    if os.path.exists(_p):
        REF = json.load(open(_p))["quantum"]; print(f"\n(loaded quantum reference from {_p})")
    else:
        print("\nNo quantum reference found (run Cell 3 or set RERUN_QUANTUM_REFERENCE=True) -> comparison skipped.")
if REF is not None:
    CMP = {"quantum (144)": REF, **{f"{v} ({ALL[v][0]['n_par']})": ALL[v] for v in ALL if v != "quantum"}}
    print("\n" + "=" * 130 + "\nCOMPARISON (test set, mean +- std over seeds)\n" + "=" * 130)
    print(f"{'variant (params)':<22}{'R2_log':>20}{'MAE_log':>20}{'MAE_pT GeV':>20}{'MSE_pT':>20}{'us/row':>12}{'sec/epoch':>11}{'ep->2%':>8}")
    for nm, rs in CMP.items():
        e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), np.nan) for r in rs]
        print(f"{nm:<22}{ms([r['test']['r2_log'] for r in rs], '{:.4f}'):>20}{ms([r['test']['mae_log'] for r in rs], '{:.4f}'):>20}"
              f"{ms([r['test']['mae_pt'] for r in rs], '{:.3f}'):>20}{ms([r['test']['mse_pt'] for r in rs], '{:.2f}'):>20}"
              f"{np.mean([r['timing']['graph']['us_per_row'] for r in rs]):>12.2f}"
              f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.nanmean(e2):>8.1f}")
    qs = {r["seed"]: r for r in REF}
    print("\nPAIRED DIFFERENCE  (quantum minus variant; for R2 positive = quantum better, for errors negative = quantum better):")
    for v in ALL:
        if v == "quantum": continue
        common = [r["seed"] for r in ALL[v] if r["seed"] in qs]
        vs = {r["seed"]: r for r in ALL[v]}
        for lab, key in [("dR2_log", "r2_log"), ("dMAE_log", "mae_log"), ("dMAE_pT", "mae_pt")]:
            d = np.array([qs[s]["test"][key] - vs[s]["test"][key] for s in common])
            wins = int((d > 0).sum()) if key == "r2_log" else int((d < 0).sum())
            print(f"   quantum - {v:<12} {lab:<9} {d.mean():+.5f} +- {d.std(ddof=1) if len(d) > 1 else 0:.5f}   quantum better in {wins}/{len(d)} seeds")
    print("\nA gap is evidence only if its mean clearly exceeds the std and the sign is the same in (nearly) all seeds.")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['classical36', 'classical16', 'noquantum']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# VARIANT: classical36   |   144 trainable parameters
################################################################################################
   node_proj 7->4                            32
   physics_prior 2->1                         3
   head 4->6                                 30
   head 6->1                                  7
   pair_proj 8->4                            36
   classical block 4-4-4 (W1,b1,W2)          36
   TOTAL                                    144

--- classical36 | seed 42 ---
[classical36 | seed 42] ep  1/50  train 0.2288  val loss 0.1675  R2_log 0.4552  MAE_log 0.3828  MAE_pT 15.217 GeV   

In [5]:
"""
================================================================================
CELL 5 -- DISTIL THE QUANTUM INTERACTION BLOCK INTO CLASSICAL PARAMETERS                         [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0, Cell 1 and Cell 3 (Cell 3 must have saved piqgat_quantum_seed{S}.npz = the trained 144-param teacher).

TEACHER  the trained PI-QGAT: node_proj, pair_proj, physics prior, head are classical; the 4-qubit circuit + entangle_gate1
         (36 params) maps theta (4 angles per edge) -> z = <Z_k> (4 values per edge).
STUDENT  the SAME network with the 36-param quantum block replaced by a classical MLP  z = tanh(W2 tanh(W1 theta + b1)),
         hidden width H.  H = 4 -> 36 block params -> 144 total (equal budget).  H = 8/16/32 -> 180/252/396 total: "how much
         classical capacity does it take to absorb the circuit?".  No circuit is evaluated at inference.

STAGE A  block-level, LABEL-FREE hint fit (FitNets-style / "classical surrogate"):  min_c  E || block_c(theta) - z_teacher(theta) ||^2
         over every edge of every training event, theta from the TEACHER's own trunk.  Result = "zero-shot swap": drop the student
         block in, change nothing else, no label ever seen.  Measures pure REPRESENTABILITY of the circuit by H hidden units.
STAGE B  prediction-level KD fine-tune of ALL student parameters (small lr):
             L = LAM_Y * SmoothL1(y_s, y_true) + LAM_T * SmoothL1(y_s, y_teacher) + LAM_Z * MSE(z_s, z_teacher(theta_student))
         (regression KD: no temperature; the teacher's log-pT output is the "soft label"; the hint term keeps the block faithful
         on the thetas the student's own trunk produces).
CONTROL  same budget, same warm-started trunk, random block, TRUE LABELS ONLY (LAM_T = LAM_Z = 0).  If distilled > control, the
         gain is the teacher's knowledge, not just the warm start.   Cell 4's classical36 (from scratch) is printed if its
         results file exists.
READ-OUT students beating/matching the teacher at the same 144 params => the circuit's value was a better OPTIMISATION PATH, not a
         better function class.  Students lagging => the circuit expresses something 36 tanh units cannot (checked by the H sweep).
================================================================================
"""
SEEDS = [42]                              # teacher seeds to distil (need piqgat_quantum_seed{S}.npz from Cell 3)
VARIANTS_TO_RUN = ["distilled"]
H_LIST = [4, 8, 16, 32]                   # student hidden widths; H = 4 is the 144-param, equal-budget student
HEADLINE_H = 4
A_EPOCHS, A_BATCH, A_LR = 40, 8192, 3e-3  # stage A (label-free block fit)
B_EPOCHS, B_LR = 20, 3e-4                 # stage B (KD fine-tune), batch = BATCH_SIZE (64), same clip / wd as Cell 3
LAM_Y, LAM_T, LAM_Z = 0.5, 0.5, 0.5       # true-label / teacher-output / block-hint weights
RUN_CONTROL = True                        # headline-size student, true labels only, warm-started trunk, random block
RESULTS_FILE = "piqgat_results_distill.json"
# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, sets the RZZ angles)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 11. DISTILLATION
# ==============================================================================
print("\n" + "#" * 96 + "\n# CELL 5: distilling the quantum block into classical parameters\n" + "#" * 96)
Ct = Compiled("quantum")
_sl1j = _sl1

def tf_from(P, x):
    h = _lin(P["node"], x)
    u = _lin(P["pair"], jnp.concatenate([h[:, RECV], h[:, SEND]], -1))
    return (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)

@jax.jit
def teacher_block(Pt, M0, M1, tf):
    a1 = jax.nn.sigmoid(_lin(Pt["gate1"], tf)) * math.pi
    return circuit_core(tf, M0, M1, a1)

def student_block(c, tf):
    return jnp.tanh(jnp.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T)

def forward_s(P, x, phi, ba):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x); h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, N_QUBITS)
    z = student_block(P["c"], tf).reshape(B, N_EDGES, N_QUBITS)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), tf, z

def student_init(Pt, H, seed, random_block=True):
    ks = jax.random.split(jax.random.PRNGKey(10_000 + seed), 2)
    l1, l2 = _init_lin(ks[0], N_QUBITS, H), _init_lin(ks[1], H, N_QUBITS)
    P = {k: Pt[k] for k in ("node", "pair", "phys", "head1", "head2")}      # trunk + head warm-started from the teacher
    P["c"] = {"W1": l1["W"], "b1": l1["b"], "W2": l2["W"]}
    return P

def n_params(P): return int(sum(a.size for a in jax.tree_util.tree_leaves(P)))

def chunked(fn, *arrs, bs=EVAL_BATCH_SIZE):
    return jnp.concatenate([fn(*[a[i:i + bs] for a in arrs]) for i in range(0, arrs[0].shape[0], bs)])

def make_adam_epoch(loss_fn, wd):
    @jax.jit
    def epoch(w, mm, vv, t, lr, batches):
        def step(carry, b):
            w, mm, vv, t = carry
            l, g = jax.value_and_grad(lambda ww: loss_fn(ww, *b))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - lr * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), l
        (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), batches)
        return w, mm, vv, t, ls.mean()
    return epoch

def np_forward(Pn, x, phi, ba):
    """Pure-NumPy student inference (no JAX, no PennyLane): this is the deployable model."""
    lin = lambda p, a: a @ p["W"].T + p["b"]
    h = lin(Pn["node"], x); h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = np.arctan2(np.sin(d), np.cos(d)) if PHI_IS_RADIANS else d
    r = np.stack([np.abs(ba[:, SEND] - ba[:, RECV]), np.abs(dphi)], -1)
    Pp = lin(Pn["phys"], (r - np.asarray(RM)) / np.asarray(RS))[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * np.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * np.tanh(lin(Pn["pair"], np.concatenate([h_i, h_j], -1)))).reshape(-1, N_QUBITS)
    c = Pn["c"]; z = np.tanh(np.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T).reshape(len(x), N_EDGES, N_QUBITS)
    s = (z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp).reshape(len(x), N_NODES, N_NODES - 1)
    e = np.exp(s - s.max(-1, keepdims=True)); alpha = e / e.sum(-1, keepdims=True)
    m = (alpha[..., None] * (h_j + z).reshape(len(x), N_NODES, N_NODES - 1, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = (np.tanh(g @ Pn["head1"]["W"].T + Pn["head1"]["b"]) @ Pn["head2"]["W"].T + Pn["head2"]["b"])[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def fidelity(zs, zt):
    zs, zt = np.asarray(zs, np.float64), np.asarray(zt, np.float64)
    return dict(z_r2=float(1 - ((zs - zt) ** 2).sum() / ((zt - zt.mean(0)) ** 2).sum()), z_rmse=float(np.sqrt(((zs - zt) ** 2).mean())),
                z_maxerr=float(np.abs(zs - zt).max()))

def eval_student(pred_fn, w, Xs, ps, bs_, y_log, pt_true, pt_teacher_log=None):
    p = np.asarray(chunked(lambda a, b, c: pred_fn(w, a, b, c), Xs, ps, bs_))
    m, pt_pred = split_metrics(p, y_log, pt_true)
    if pt_teacher_log is not None:
        m["agree_r2_vs_teacher"] = float(1 - ((p - pt_teacher_log) ** 2).sum() / ((pt_teacher_log - pt_teacher_log.mean()) ** 2).sum())
        m["agree_mae_vs_teacher"] = float(np.abs(p - pt_teacher_log).mean())
    return m, p, pt_pred

def distil_one(seed):
    f = os.path.join(RESULTS_DIR, f"piqgat_quantum_seed{seed}.npz")
    assert os.path.isfile(f), f"{f} not found -- run Cell 3 with this seed first"
    wt = jnp.asarray(np.load(f)["w"]); Pt = Ct.unravel(wt); M01 = Ct.fold(wt)
    # ---- teacher reference ----
    yT_tr = np.asarray(predict_log(Ct, wt, Xtr, ptr, btr, M01)); yT_va = predict_log(Ct, wt, Xva, pva, bva, M01)
    yT_te = predict_log(Ct, wt, Xte, pte, bte, M01)
    mT, ptT = split_metrics(yT_te, y_test, pT_test64); binsT = bin_stats(pT_test64, ptT, yT_te, y_test)
    print(f"\n[teacher seed {seed}] {n_params(Pt)} params | TEST R2_log {mT['r2_log']:.4f}  MAE_log {mT['mae_log']:.4f}  MAE_pT {mT['mae_pt']:.3f} GeV")
    # ---- theta / z_teacher on every edge (teacher trunk), train + val ----
    tfj = jax.jit(tf_from)
    TF = chunked(lambda x: tfj(Pt, x), Xtr); ZT = chunked(lambda t: teacher_block(Pt, M01[0], M01[1], t), TF)
    TFv = chunked(lambda x: tfj(Pt, x), Xva); ZTv = chunked(lambda t: teacher_block(Pt, M01[0], M01[1], t), TFv)
    print(f"   edge samples for the block fit: {TF.shape[0]:,} train / {TFv.shape[0]:,} val   (z std per qubit: "
          f"{np.round(np.asarray(ZT.std(0)), 3).tolist()})")
    out = dict(seed=seed, teacher=dict(test=mT, bins=binsT, n_par=n_params(Pt)), students={})
    rng = np.random.default_rng(seed)
    plans = [(H, "distilled") for H in H_LIST] + ([(HEADLINE_H, "control")] if RUN_CONTROL else [])
    for H, kind in plans:
        P0 = student_init(Pt, H, seed); npar = n_params(P0); name = f"{kind}_H{H}"
        print(f"\n--- student {name}: {npar} params (block {n_params(P0['c'])}) ---")
        wS, unr = ravel_pytree(P0)
        pred_s = jax.jit(lambda w, x, p, b: forward_s(unr(w), x, p, b)[0])
        rec = dict(H=H, kind=kind, n_par=npar)
        if kind == "distilled":
            # ---------------- STAGE A: label-free block fit ----------------
            c0 = P0["c"]; wc, unr_c = ravel_pytree(c0)
            epA = make_adam_epoch(lambda w, t, z: jnp.mean((student_block(unr_c(w), t) - z) ** 2), 0.0)
            mm, vv, tt = jnp.zeros_like(wc), jnp.zeros_like(wc), jnp.float32(0.0)
            nbA = TF.shape[0] // A_BATCH; tA = time.time()
            for ep in range(1, A_EPOCHS + 1):
                perm = jnp.asarray(rng.permutation(TF.shape[0]))[:nbA * A_BATCH]
                lr = A_LR * (1 - 0.9 * (ep - 1) / max(A_EPOCHS - 1, 1))
                wc, mm, vv, tt, l = epA(wc, mm, vv, tt, lr, (TF[perm].reshape(nbA, A_BATCH, N_QUBITS), ZT[perm].reshape(nbA, A_BATCH, N_QUBITS)))
                if ep in (1, 5, 10, 20, A_EPOCHS):
                    fv = fidelity(student_block(unr_c(wc), TFv), ZTv)
                    print(f"   [A] ep {ep:2d}/{A_EPOCHS}  block-fit MSE {float(l):.5f} | val z-R2 {fv['z_r2']:.4f}  z-RMSE {fv['z_rmse']:.4f}  max|dz| {fv['z_maxerr']:.3f}", flush=True)
            P0 = dict(P0); P0["c"] = unr_c(wc); wS, unr = ravel_pytree(P0)
            pred_s = jax.jit(lambda w, x, p, b: forward_s(unr(w), x, p, b)[0])
            rec["stageA_secs"] = time.time() - tA; rec["fid_val"] = fidelity(student_block(P0["c"], TFv), ZTv)
            mA, _, _ = eval_student(pred_s, wS, Xte, pte, bte, y_test, pT_test64, yT_te)
            rec["zero_shot_test"] = mA
            print(f"   [A] ZERO-SHOT swap (no labels, nothing else changed): TEST R2_log {mA['r2_log']:.4f}  MAE_log {mA['mae_log']:.4f}  "
                  f"| agreement with teacher R2 {mA['agree_r2_vs_teacher']:.4f}")
        # ---------------- STAGE B: KD fine-tune (all params) ----------------
        lam_y, lam_t, lam_z = (LAM_Y, LAM_T, LAM_Z) if kind == "distilled" else (1.0, 0.0, 0.0)
        def loss_b(w, x, p, b, y, yT):
            ys, tf, zs = forward_s(unr(w), x, p, b)
            l = lam_y * _sl1j(ys, y)
            if lam_t: l = l + lam_t * _sl1j(ys, yT)
            if lam_z:
                zt = jax.lax.stop_gradient(teacher_block(Pt, M01[0], M01[1], tf))
                l = l + lam_z * jnp.mean((zs.reshape(-1, N_QUBITS) - zt) ** 2)
            return l
        epB = make_adam_epoch(loss_b, CLASSICAL_WEIGHT_DECAY)
        mm, vv, tt = jnp.zeros_like(wS), jnp.zeros_like(wS), jnp.float32(0.0)
        nfull = NB * BATCH_SIZE; shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
        yT_trj = jnp.asarray(yT_tr, jnp.float32)
        pv0 = np.asarray(chunked(lambda a, b, c: pred_s(wS, a, b, c), Xva, pva, bva))
        best, best_w, best_ep, hist = split_metrics(pv0, y_val, pT_val)[0]["loss"], wS, 0, []
        tB = time.time()
        for ep in range(1, B_EPOCHS + 1):
            t0 = time.time(); perm = jnp.asarray(rng.permutation(N_TR))
            wS, mm, vv, tt, tl = epB(wS, mm, vv, tt, B_LR, (shp(Xtr[perm]), shp(ptr[perm]), shp(btr[perm]), shp(ytr[perm]), shp(yT_trj[perm])))
            pv = np.asarray(chunked(lambda a, b, c: pred_s(wS, a, b, c), Xva, pva, bva))
            vm, _ = split_metrics(pv, y_val, pT_val)
            flag = ""
            if vm["loss"] < best - 1e-6: best, best_w, best_ep, flag = vm["loss"], jnp.array(wS), ep, " *best"
            hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], sec=time.time() - t0))
            print(f"   [B {name}] ep {ep:2d}/{B_EPOCHS}  loss {float(tl):.4f}  val loss {vm['loss']:.4f}  R2_log {vm['r2_log']:.4f}  {time.time()-t0:4.1f}s{flag}", flush=True)
        # (best_ep 0 = the stage-A / warm-start state itself was best)
        wS = best_w
        m, p, pt_pred = eval_student(pred_s, wS, Xte, pte, bte, y_test, pT_test64, yT_te)
        rec.update(best_ep=best_ep, test=m, bins=bin_stats(pT_test64, pt_pred, p, y_test), hist=hist, stageB_secs=time.time() - tB)
        print(f"   >> {name} | best ep {best_ep} | TEST R2_log {m['r2_log']:.4f}  MAE_log {m['mae_log']:.4f}  MAE_pT {m['mae_pt']:.3f} GeV "
              f"| vs teacher: dR2_log {m['r2_log']-mT['r2_log']:+.4f}  agreement R2 {m['agree_r2_vs_teacher']:.4f}")
        # timing: full test pass
        full = lambda: chunked(lambda a, b, c: pred_s(wS, a, b, c), Xte, pte, bte).block_until_ready()
        full(); rec["us_per_row"] = 1e6 * _med(full, 3) / Xte.shape[0]
        # pure-NumPy deployable check
        Pn = jax.tree_util.tree_map(lambda a: np.asarray(a, np.float64), unr(wS))
        npred = np_forward(Pn, np.asarray(Xte[:20000], np.float64), np.asarray(pte[:20000], np.float64), np.asarray(bte[:20000], np.float64))
        rec["numpy_maxdiff"] = float(np.abs(npred - p[:20000]).max())
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_student_{name}_seed{seed}.npz"), w=np.asarray(wS), w_fp16=np.asarray(wS.astype(jnp.float16)))
        out["students"][name] = rec
    # teacher timing for the table
    Cf = lambda: chunked(lambda a, b, c: Ct.pred(wt, M01, a, b, c), Xte, pte, bte).block_until_ready()
    Cf(); out["teacher"]["us_per_row"] = 1e6 * _med(Cf, 3) / Xte.shape[0]
    return out

ALLD = [distil_one(s) for s in SEEDS]

# ==============================================================================
# 12. REPORT
# ==============================================================================
ref = {}
_cf = os.path.join(RESULTS_DIR, "piqgat_results_ablation.json")
if os.path.isfile(_cf):
    for v, rs in json.load(open(_cf)).items():
        ref[v] = {r["seed"]: r["test"] for r in rs}
print("\n" + "=" * 118 + f"\nFINAL REPORT | quantum-block distillation | seeds {SEEDS} | TEST set, log1p(pT) scale; mean +- std over seeds\n" + "=" * 118)
print(f"{'model':<26}{'params':>8}{'R2_log':>18}{'MAE_log':>18}{'MAE_pT (GeV)':>18}{'dR2 vs teacher':>16}{'agree R2':>10}{'us/row':>8}")
def row(lab, npar, ms_list, us, tR2=None, agree=None):
    r2 = [m["r2_log"] for m in ms_list]
    d = f"{np.mean(np.array(r2) - np.array(tR2)):+.4f}" if tR2 is not None else "-"
    ag = f"{np.mean([m['agree_r2_vs_teacher'] for m in ms_list]):.4f}" if agree else "-"
    print(f"{lab:<26}{npar:>8}{ms(r2, '{:.4f}'):>18}{ms([m['mae_log'] for m in ms_list], '{:.4f}'):>18}"
          f"{ms([m['mae_pt'] for m in ms_list], '{:.3f}'):>18}{d:>16}{ag:>10}{us:>8.2f}")
tR2 = [d["teacher"]["test"]["r2_log"] for d in ALLD]
row("TEACHER (quantum)", ALLD[0]["teacher"]["n_par"], [d["teacher"]["test"] for d in ALLD], np.mean([d["teacher"]["us_per_row"] for d in ALLD]), None)
for H in H_LIST:
    nm = f"distilled_H{H}"
    rs = [d["students"][nm] for d in ALLD]
    row(f"  A only (zero-shot) H={H}", rs[0]["n_par"], [r["zero_shot_test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
    row(f"  A+B distilled   H={H}", rs[0]["n_par"], [r["test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
if RUN_CONTROL:
    rs = [d["students"][f"control_H{HEADLINE_H}"] for d in ALLD]
    row(f"  CONTROL (labels) H={HEADLINE_H}", rs[0]["n_par"], [r["test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
for v in ("classical36", "classical16", "noquantum"):
    if v in ref:
        ms_ = [ref[v][s] for s in SEEDS if s in ref[v]]
        if ms_: print(f"{'  Cell 4 ' + v + ' (scratch)':<26}{'':>8}{ms([m['r2_log'] for m in ms_], '{:.4f}'):>18}{ms([m['mae_log'] for m in ms_], '{:.4f}'):>18}"
                      f"{ms([m['mae_pt'] for m in ms_], '{:.3f}'):>18}{'':>16}{'':>10}{'':>8}")

print("\nBLOCK FIDELITY after stage A (validation edges):")
for H in H_LIST:
    fv = [d["students"][f"distilled_H{H}"]["fid_val"] for d in ALLD]
    print(f"   H={H:<3} z-R2 {ms([f['z_r2'] for f in fv], '{:.4f}')}   z-RMSE {ms([f['z_rmse'] for f in fv], '{:.4f}')}   max|dz| {ms([f['z_maxerr'] for f in fv], '{:.3f}')}")

print(f"\nPER pT CLASS, test (seed-mean): teacher vs distilled H={HEADLINE_H}" + (" vs control" if RUN_CONTROL else ""))
print(f"{'pT class (GeV)':>16}{'N':>9}{'MAE_pT teach':>14}{'MAE_pT dist':>13}" + (f"{'MAE_pT ctrl':>13}" if RUN_CONTROL else "") + f"{'MAE_log teach':>15}{'MAE_log dist':>14}")
for i, b0 in enumerate(ALLD[0]["teacher"]["bins"]):
    g = lambda get: np.mean([get(d, i) for d in ALLD])
    line = (f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{g(lambda d, i: d['teacher']['bins'][i]['mae_pt']):>14.3f}"
            f"{g(lambda d, i: d['students'][f'distilled_H{HEADLINE_H}']['bins'][i]['mae_pt']):>13.3f}")
    if RUN_CONTROL: line += f"{g(lambda d, i: d['students'][f'control_H{HEADLINE_H}']['bins'][i]['mae_pt']):>13.3f}"
    line += (f"{g(lambda d, i: d['teacher']['bins'][i]['mae_log']):>15.4f}"
             f"{g(lambda d, i: d['students'][f'distilled_H{HEADLINE_H}']['bins'][i]['mae_log']):>14.4f}")
    print(line)
hs = [d["students"][f"distilled_H{HEADLINE_H}"] for d in ALLD]
print(f"\nDEPLOYMENT: the H={HEADLINE_H} student is {hs[0]['n_par']} params, pure tanh/linear/softmax; pure-NumPy re-implementation max|diff| vs JAX "
      f"{max(h['numpy_maxdiff'] for h in hs):.1e}. Inference {np.mean([h['us_per_row'] for h in hs]):.2f} us/row vs teacher "
      f"{np.mean([d['teacher']['us_per_row'] for d in ALLD]):.2f} us/row (folded circuit, same device, same batch size).")
print("HOW TO READ: A-only = pure representability of the circuit; A+B vs CONTROL = value of the teacher's knowledge beyond a warm-started trunk;\n"
      "             A+B vs teacher = accuracy cost (or gain) of removing the circuit; H sweep = classical capacity needed to absorb it.")

_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(ALLD, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['distilled']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# CELL 5: distilling the quantum block into classical parameters
################################################################################################

[teacher seed 42] 144 params | TEST R2_log 0.8395  MAE_log 0.2143  MAE_pT 11.852 GeV
   edge samples for the block fit: 2,830,452 train / 2,830,452 val   (z std per qubit: [0.2930000126361847, 0.7929999828338623, 0.5189999938011169, 0.734000027179718])

--- student distilled_H4: 144 params (block 36) ---
   [A] ep  1/40  block-fit MSE 0.27634 | val z-R2 0.5677  z-RMSE 0.4055  max|dz| 1.720
   [A] ep  5/40  block-fit MSE 0.11628 | val z-R2 0.6987  z-RMSE 0.3386  max|dz| 1.877
   [A] ep 10/40  bloc